# Opponent Net Rating vs Player Points

This notebook explores whether there is a relationship between:

- a player's points in a game
- the opponent team's rolling net rating over its previous games
- and the player's own rolling scoring baseline

Default setup:

- opponent team rolling net rating: previous `20` team games
- player rolling average points: previous `40` player games

The workflow is:

1. build one row per player-game
2. attach the opponent's rolling net rating
3. attach the player's rolling points average
4. analyze overall relationships and player-level relationships


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

PROJECT_ROOT = Path.cwd()
PLAYERS_PATH = PROJECT_ROOT / "data/box_scores/players.csv"
TEAMS_PATH = PROJECT_ROOT / "data/box_scores/teams.csv"

TEAM_WINDOW = 20
PLAYER_WINDOW = 40

MIN_TEAM_HISTORY = TEAM_WINDOW
MIN_PLAYER_HISTORY = PLAYER_WINDOW


In [ ]:
def load_data(players_path=PLAYERS_PATH, teams_path=TEAMS_PATH):
    players = pd.read_csv(players_path, dtype={"gameId": str, "teamId": str, "personId": str})
    teams = pd.read_csv(teams_path, dtype={"gameId": str, "teamId": str})

    for df in (players, teams):
        df["gameId"] = df["gameId"].str.lstrip("0")
        df["gameDate"] = pd.to_datetime(df["gameDate"])

    return players, teams


def build_analysis_frame(team_window=TEAM_WINDOW, player_window=PLAYER_WINDOW,
                         min_team_history=MIN_TEAM_HISTORY,
                         min_player_history=MIN_PLAYER_HISTORY):
    players, teams = load_data()

    teams = teams.sort_values(["teamId", "gameDate", "gameId"]).copy()
    teams["opponent_net_rating_roll"] = (
        teams.groupby("teamId")["netRating"]
        .transform(lambda s: s.shift(1).rolling(team_window, min_periods=min_team_history).mean())
    )

    opponent_lookup = teams[["gameId", "teamId", "teamTricode", "netRating", "opponent_net_rating_roll"]].rename(
        columns={
            "teamId": "opponentTeamId",
            "teamTricode": "opponentTeamTricode",
            "netRating": "opponentNetRating_game",
        }
    )

    analysis = players.merge(opponent_lookup, on="gameId", how="left")
    analysis = analysis[analysis["teamId"] != analysis["opponentTeamId"]].copy()

    analysis = analysis.sort_values(["personId", "gameDate", "gameId"]).copy()
    analysis["player_points_roll"] = (
        analysis.groupby("personId")["points"]
        .transform(lambda s: s.shift(1).rolling(player_window, min_periods=min_player_history).mean())
    )

    analysis["player_name"] = analysis["firstName"] + " " + analysis["familyName"]
    analysis["points_minus_player_roll"] = analysis["points"] - analysis["player_points_roll"]
    analysis["season"] = analysis["gameDate"].apply(lambda d: f"{d.year}-{str(d.year + 1)[2:]}" if d.month >= 7 else f"{d.year - 1}-{str(d.year)[2:]}")

    return analysis


In [ ]:
analysis = build_analysis_frame()
analysis_ready = analysis.dropna(subset=["opponent_net_rating_roll", "player_points_roll"]).copy()

print("All player-game rows:", len(analysis))
print("Rows with full rolling history:", len(analysis_ready))
print("Unique players with full rolling history:", analysis_ready["personId"].nunique())

analysis_ready[[
    "gameDate",
    "season",
    "player_name",
    "teamTricode",
    "opponentTeamTricode",
    "points",
    "player_points_roll",
    "points_minus_player_roll",
    "opponent_net_rating_roll",
]].head()


In [ ]:
overall_corr_points = analysis_ready["points"].corr(analysis_ready["opponent_net_rating_roll"])
overall_corr_residual = analysis_ready["points_minus_player_roll"].corr(analysis_ready["opponent_net_rating_roll"])

print(f"Correlation: points vs opponent rolling net rating = {overall_corr_points:.4f}")
print(f"Correlation: (points - player rolling average) vs opponent rolling net rating = {overall_corr_residual:.4f}")


In [ ]:
decile_view = analysis_ready.copy()
decile_view["opp_net_rating_decile"] = pd.qcut(decile_view["opponent_net_rating_roll"], q=10, duplicates="drop")

decile_summary = (
    decile_view.groupby("opp_net_rating_decile", observed=False)
    .agg(
        rows=("gameId", "size"),
        mean_opp_net_rating=("opponent_net_rating_roll", "mean"),
        mean_points=("points", "mean"),
        mean_player_roll=("player_points_roll", "mean"),
        mean_points_minus_roll=("points_minus_player_roll", "mean"),
    )
    .reset_index()
)

decile_summary


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharex=True)

axes[0].plot(decile_summary["mean_opp_net_rating"], decile_summary["mean_points"], marker="o")
axes[0].set_title("Actual Points vs Opponent Rolling Net Rating")
axes[0].set_xlabel("Opponent Rolling Net Rating")
axes[0].set_ylabel("Mean Actual Points")

axes[1].plot(decile_summary["mean_opp_net_rating"], decile_summary["mean_points_minus_roll"], marker="o")
axes[1].axhline(0.0, color="black", linewidth=1, linestyle="--")
axes[1].set_title("Residual Points vs Opponent Rolling Net Rating")
axes[1].set_xlabel("Opponent Rolling Net Rating")
axes[1].set_ylabel("Mean (Points - Player Rolling Avg)")

plt.tight_layout()


In [ ]:
def player_corr_summary(df, min_games=60, min_avg_points=10.0):
    grouped = []
    for (person_id, player_name), player_df in df.groupby(["personId", "player_name"]):
        player_df = player_df.sort_values(["gameDate", "gameId"])
        if len(player_df) < min_games:
            continue
        if player_df["points"].mean() < min_avg_points:
            continue

        corr_points = player_df["points"].corr(player_df["opponent_net_rating_roll"])
        corr_residual = player_df["points_minus_player_roll"].corr(player_df["opponent_net_rating_roll"])

        grouped.append(
            {
                "personId": person_id,
                "player_name": player_name,
                "games": len(player_df),
                "avg_points": player_df["points"].mean(),
                "corr_points": corr_points,
                "corr_residual": corr_residual,
            }
        )

    return pd.DataFrame(grouped)


player_summary = player_corr_summary(analysis_ready, min_games=60, min_avg_points=10.0)
print("Players in summary:", len(player_summary))
player_summary.head()


In [ ]:
print("Top positive residual correlations")
display(player_summary.sort_values("corr_residual", ascending=False).head(15))

print("Top negative residual correlations")
display(player_summary.sort_values("corr_residual", ascending=True).head(15))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.hist(player_summary["corr_residual"].dropna(), bins=40, edgecolor="white")
ax.axvline(player_summary["corr_residual"].median(), color="red", linestyle="--", label="median")
ax.set_title("Distribution of Player-Level Residual Correlations")
ax.set_xlabel("corr(points - player rolling avg, opponent rolling defensive rating)")
ax.set_ylabel("Player count")
ax.legend()
plt.tight_layout()


## Notes

- A positive correlation here means the player tends to score **more** when facing teams with a **higher** rolling net rating.
- Since higher net rating generally corresponds to stronger teams overall, the sign here is less straightforward than defensive rating and is worth interpreting carefully.
- This notebook is exploratory, not causal. It is mainly useful for checking whether opponent defense looks informative enough to turn into a feature later.
- Easy follow-ups:
  - vary the team/player windows
  - restrict to starters or higher-minute players
  - analyze by season
  - fit a simple linear regression on `points_minus_player_roll ~ opponent_net_rating_roll`
